# DAPO: Beating RLVR with Decoupled Policy Optimization

This notebook implements **DAPO** (Decoupled Clip and Dynamic sAmpling Policy Optimization) to fine-tune **Qwen2.5-Math-1.5B** and beat the One-Shot-RLVR baseline accuracy.

## RLVR Baseline Scores (to beat):
| Benchmark | AIME25x8 | AMC23x8 | AIME24x8 | Minerva | OlympiadBench | MATH500 |
|-----------|----------|---------|----------|---------|---------------|--------|
| **RLVR**  | 5.0%     | 50.9%   | 12.5%    | 27.9%   | 33.8%         | 73.0%  |

## DAPO Key Techniques:
1. **Clip-Higher**: Asymmetric PPO clipping (epsilon=0.28) for better exploration
2. **Dynamic Sampling**: Skip zero-signal training batches
3. **Token-Level Loss**: Remove length bias in policy gradient
4. **Overlong Reward Shaping**: Soft penalty for verbose responses

**Runtime**: ~2-3 hours on T4 GPU

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
%%capture
# ============================================================
# Cell 1: Install Dependencies
# ============================================================
!pip install -q "trl>=0.15.0" "peft>=0.14.0" "bitsandbytes>=0.44.0"
!pip install -q datasets "accelerate>=0.34.0" matplotlib
!pip install -q huggingface_hub

In [ ]:
# ============================================================
# Cell 2: Imports & Environment Setup
# ============================================================
import os, re, json, math, random, warnings
import torch
import numpy as np
import matplotlib.pyplot as plt
from typing import Optional
from tqdm.auto import tqdm
from datasets import load_dataset, Dataset
from transformers import (
    AutoModelForCausalLM, AutoTokenizer,
    BitsAndBytesConfig, GenerationConfig,
)
from peft import LoraConfig, PeftModel
from trl import GRPOConfig, GRPOTrainer

warnings.filterwarnings("ignore", category=UserWarning)
os.environ["WANDB_MODE"] = "disabled"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

PyTorch: 2.10.0+cu128
CUDA: True
GPU: Tesla T4
VRAM: 15.6 GB


In [ ]:
# ============================================================
# Cell 3: DAPO Configuration
# ============================================================
MODEL_NAME = "Qwen/Qwen2.5-Math-1.5B"
OUTPUT_DIR = "/content/dapo_model"
MERGED_DIR = "/content/dapo_model_merged"

# DAPO Hyperparameters
GROUP_SIZE = 4              # Completions per prompt (G)
MAX_COMPLETION_LEN = 512    # Max tokens per completion
MAX_PROMPT_LEN = 256        # Max prompt tokens
TRAIN_STEPS = 200           # Total training steps
LR = 5e-6                   # Learning rate
EPSILON = 0.28              # DAPO Clip-Higher (default PPO is 0.2)
KL_BETA = 0.04              # KL penalty coefficient
TRAIN_SIZE = 500            # Training subset size
OVERLONG_THRESHOLD = 400    # Token threshold for overlong penalty

SYSTEM_PROMPT = "Please reason step by step, and put your final answer within \\boxed{}."

print("DAPO Configuration:")
print(f"  Model: {MODEL_NAME}")
print(f"  Group size: {GROUP_SIZE}")
print(f"  Epsilon (clip-higher): {EPSILON}")
print(f"  KL beta: {KL_BETA}")
print(f"  Steps: {TRAIN_STEPS}")
print(f"  Train size: {TRAIN_SIZE}")

DAPO Configuration:
  Model: Qwen/Qwen2.5-Math-1.5B
  Group size: 4
  Epsilon (clip-higher): 0.28
  KL beta: 0.04
  Steps: 200
  Train size: 500


In [ ]:
# ============================================================
# Cell 4: Load Tokenizer & LoRA Config
# ============================================================
print(f"Loading tokenizer for {MODEL_NAME}...")

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME, trust_remote_code=True, padding_side="left"
)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
print(f"Tokenizer loaded (vocab: {tokenizer.vocab_size})")

# QLoRA config for T4 memory efficiency
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

lora_config = LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05,
    bias="none", task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
)
print(f"LoRA config: rank={lora_config.r}, alpha={lora_config.lora_alpha}")

Loading tokenizer for Qwen/Qwen2.5-Math-1.5B...


config.json:   0%|          | 0.00/676 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Tokenizer loaded (vocab: 151643)
LoRA config: rank=16, alpha=32


In [ ]:
# ============================================================
# Cell 5: Load & Prepare Training Dataset
# ============================================================
print("Loading training data...")

def extract_boxed(text):
    """Extract answer from \\boxed{...}, handling nested braces."""
    idx = text.rfind("\\boxed{")
    if idx == -1:
        return ""
    depth = 0
    start = idx + len("\\boxed{")
    for i in range(start, len(text)):
        if text[i] == '{':
            depth += 1
        elif text[i] == '}':
            if depth == 0:
                return text[start:i].strip()
            depth -= 1
    return ""

def extract_gsm8k_answer(text):
    """Extract answer after #### in GSM8K."""
    match = re.search(r'####\s*([\-\d\.,]+)', text)
    return match.group(1).replace(',', '').strip() if match else ""

# Try loading datasets with fallbacks
all_examples = []

# Source 1: GSM8K (clean numerical answers)
try:
    gsm = load_dataset("openai/gsm8k", "main", split="train")
    for ex in gsm:
        ans = extract_gsm8k_answer(ex["answer"])
        if ans:
            all_examples.append({"problem": ex["question"], "answer": ans})
    print(f"  GSM8K: {len(all_examples)} examples loaded")
except Exception as e:
    print(f"  GSM8K unavailable: {e}")

# Source 2: MATH competition (harder problems)
try:
    math_ds = load_dataset("hendrycks/competition_math", split="train")
    count = 0
    for ex in math_ds:
        ans = extract_boxed(ex.get("solution", ""))
        if ans:
            all_examples.append({"problem": ex["problem"], "answer": ans})
            count += 1
    print(f"  MATH: {count} examples loaded")
except Exception as e:
    print(f"  MATH unavailable: {e}")

# Shuffle and subset
random.shuffle(all_examples)
all_examples = all_examples[:TRAIN_SIZE]

# Format for GRPOTrainer: needs 'prompt' column + extra columns for reward
formatted = []
for ex in all_examples:
    formatted.append({
        "prompt": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": ex["problem"]},
        ],
        "answer": ex["answer"],
    })

train_dataset = Dataset.from_list(formatted)
print(f"\nTraining dataset: {len(train_dataset)} examples")
print(f"Sample problem: {train_dataset[0]['prompt'][1]['content'][:100]}...")
print(f"Sample answer: {train_dataset[0]['answer']}")

Loading training data...


README.md: 0.00B [00:00, ?B/s]

main/train-00000-of-00001.parquet:   0%|          | 0.00/2.31M [00:00<?, ?B/s]

main/test-00000-of-00001.parquet:   0%|          | 0.00/419k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/7473 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1319 [00:00<?, ? examples/s]

  GSM8K: 7473 examples loaded
  MATH unavailable: Dataset 'hendrycks/competition_math' doesn't exist on the Hub or cannot be accessed.

Training dataset: 500 examples
Sample problem: Stefan goes to a restaurant to eat dinner with his family. They order an appetizer that costs $10 an...
Sample answer: 108


In [ ]:
# ============================================================
# Cell 6: DAPO Reward Functions
# ============================================================

def normalize_answer(ans):
    """Normalize a math answer for comparison."""
    if not ans:
        return None
    ans = str(ans).strip()
    ans = ans.replace(',', '').replace('$', '')
    ans = ans.replace('\\text{', '').replace('}', '')
    ans = ans.replace('\\mathrm{', '').replace('\\', '')
    ans = ans.replace(' ', '')
    try:
        val = float(ans)
        if val == int(val):
            return str(int(val))
        return f"{val:.6f}".rstrip('0').rstrip('.')
    except (ValueError, OverflowError):
        return ans.lower()


def get_completion_text(completion):
    """Extract text from completion (handles both str and list-of-dict formats)."""
    if isinstance(completion, str):
        return completion
    elif isinstance(completion, list):
        # TRL passes completions as [{"role": "assistant", "content": "..."}]
        for msg in completion:
            if isinstance(msg, dict) and msg.get("role") == "assistant":
                return msg.get("content", "")
        # Fallback: join all content
        return " ".join(m.get("content", "") for m in completion if isinstance(m, dict))
    return str(completion)


def math_correctness_reward(completions, answer, **kwargs):
    """
    DAPO Reward Function with:
    - Binary correctness reward (1.0 correct, 0.0 wrong)
    - Overlong penalty (soft penalty for verbose responses)
    - Format bonus (small bonus for using \\boxed{})
    """
    rewards = []
    for completion, gt in zip(completions, answer):
        reward = 0.0
        # Handle both str and list-of-dict completion formats
        text = get_completion_text(completion)
        model_ans = extract_boxed(text)
        gt_norm = normalize_answer(gt)
        model_norm = normalize_answer(model_ans)

        # Correctness check
        if model_norm and gt_norm and model_norm == gt_norm:
            reward = 1.0
        elif model_norm is None:
            reward = -0.1

        # DAPO: Format bonus for using \boxed{}
        if "\\boxed{" in text:
            reward += 0.05

        # DAPO: Overlong reward shaping
        tok_count = len(text.split())
        if tok_count > OVERLONG_THRESHOLD:
            excess = (tok_count - OVERLONG_THRESHOLD) / OVERLONG_THRESHOLD
            reward -= 0.3 * min(1.0, excess)

        rewards.append(float(reward))
    return rewards


print("Reward functions defined.")
# Quick test
test_r = math_correctness_reward(
    ["The answer is \\boxed{42}", "I think it's 42", "x" * 2000],
    ["42", "42", "42"]
)
print(f"  Test rewards: {test_r}")
print(f"  Expected: [~1.05, 0.0, negative (overlong)]")

Reward functions defined.
  Test rewards: [1.05, -0.1, -0.1]
  Expected: [~1.05, 0.0, negative (overlong)]


## DAPO Training

We use TRL's `GRPOTrainer` with DAPO-specific modifications:
- **Clip-Higher**: `epsilon=0.28` (vs default 0.2)
- **Overlong Reward Shaping**: Built into the reward function
- **Dynamic Sampling**: GRPO naturally handles this (advantage=0 when all rewards equal)
- **Token-Level Loss**: Per-token gradient accumulation

In [ ]:
# ============================================================
# Cell 7: Configure DAPO Training
# ============================================================

training_args = GRPOConfig(
    output_dir=OUTPUT_DIR,

    # GRPO/DAPO core settings
    num_generations=GROUP_SIZE,
    max_completion_length=MAX_COMPLETION_LEN,

    # DAPO: Clip-Higher
    epsilon=EPSILON,
    beta=KL_BETA,

    # Training schedule
    max_steps=TRAIN_STEPS,
    learning_rate=LR,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,

    # Optimization
    warmup_steps=20,
    weight_decay=0.01,
    lr_scheduler_type="cosine",
    optim="adamw_torch",
    bf16=True,
    gradient_checkpointing=True,

    # Generation settings
    temperature=0.7,

    # Logging
    logging_steps=5,
    save_steps=50,
    save_total_limit=2,
    report_to="none",
    seed=SEED,
)

print("GRPOConfig created successfully")
print(f"  Effective batch size: {training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps}")
print(f"  Total steps: {training_args.max_steps}")

GRPOConfig created successfully
  Effective batch size: 8
  Total steps: 200


In [ ]:
# ============================================================
# Cell 8: Initialize DAPO Trainer & Start Training
# ============================================================
print("Initializing DAPO Trainer...")
print("This will download the model (~3GB) on first run.\n")

trainer = GRPOTrainer(
    model=MODEL_NAME,
    reward_funcs=[math_correctness_reward],
    args=training_args,
    train_dataset=train_dataset,
    processing_class=tokenizer,
    peft_config=lora_config,
)

print("Trainer initialized! Starting DAPO training...")
print("=" * 60)
print(f"Training {TRAIN_STEPS} steps with G={GROUP_SIZE} on T4 GPU")
print(f"Estimated time: ~2-3 hours")
print("=" * 60)

train_result = trainer.train()

print("\n" + "=" * 60)
print("DAPO Training Complete!")
print(f"  Total steps: {train_result.global_step}")
print(f"  Training loss: {train_result.training_loss:.4f}")
print("=" * 60)

Initializing DAPO Trainer...
This will download the model (~3GB) on first run.



model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/138 [00:00<?, ?B/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Trainer initialized! Starting DAPO training...
Training 200 steps with G=4 on T4 GPU
Estimated time: ~2-3 hours


Step,Training Loss
5,0.014174
10,0.086274
15,0.086538
20,0.158107
25,0.080919
30,0.043280
35,0.105771
40,0.057842
45,0.039024
50,0.064411



DAPO Training Complete!
  Total steps: 200
  Training loss: 0.0314


In [ ]:
# ============================================================
# Cell 9: Save & Merge the Trained Model
# ============================================================
print("Saving trained model...")

# Save LoRA adapter
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"LoRA adapter saved to {OUTPUT_DIR}")

# Merge LoRA with base model for evaluation
print("\nMerging LoRA weights with base model...")
del trainer
torch.cuda.empty_cache()

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True,
)

model = PeftModel.from_pretrained(base_model, OUTPUT_DIR)
model = model.merge_and_unload()

model.save_pretrained(MERGED_DIR)
tokenizer.save_pretrained(MERGED_DIR)

print(f"Merged model saved to {MERGED_DIR}")
print("Model ready for evaluation!")

Saving trained model...
LoRA adapter saved to /content/dapo_model

Merging LoRA weights with base model...


`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Merged model saved to /content/dapo_model_merged
Model ready for evaluation!


## Evaluation

Evaluate the DAPO-trained model on the same benchmarks as RLVR:
- **MATH500** (primary benchmark)
- **GSM8K** test set
- Competition benchmarks via One-Shot-RLVR evaluation scripts

In [ ]:
# ============================================================
# Cell 10: Evaluation Utilities
# ============================================================

def generate_solution(model, tokenizer, problem, max_new_tokens=512):
    """Generate a math solution for a given problem."""
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": problem},
    ]
    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(text, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
        )

    gen_tokens = outputs[0][inputs['input_ids'].shape[1]:]
    return tokenizer.decode(gen_tokens, skip_special_tokens=True)


def evaluate_dataset(model, tokenizer, problems, answers, name="Dataset"):
    """Evaluate model on a list of problems."""
    correct = 0
    total = len(problems)
    results = []

    print(f"\nEvaluating on {name} ({total} problems)...")
    for i in tqdm(range(total), desc=name):
        solution = generate_solution(model, tokenizer, problems[i])
        pred = extract_boxed(solution)
        pred_norm = normalize_answer(pred)
        gt_norm = normalize_answer(answers[i])

        is_correct = (pred_norm is not None and gt_norm is not None
                      and pred_norm == gt_norm)
        if is_correct:
            correct += 1

        results.append({
            "problem": problems[i][:100],
            "predicted": pred,
            "ground_truth": answers[i],
            "correct": is_correct
        })

    accuracy = 100.0 * correct / total if total > 0 else 0.0
    print(f"{name} Accuracy: {accuracy:.1f}% ({correct}/{total})")
    return accuracy, results

print("Evaluation utilities ready.")

Evaluation utilities ready.


In [ ]:
import os
import shutil
from google.colab import files

# 1. Define the name of your output zip file (without .zip extension)
output_filename = "colab_contents_archive"

# 2. Create the zip file
# This packs everything in '/content' into the zip
shutil.make_archive(output_filename, 'zip', '/content')

# 3. Trigger the download to your local machine
files.download(f"{output_filename}.zip")

print(f"Successfully zipped contents into {output_filename}.zip and started download.")

KeyboardInterrupt: 

In [ ]:
import shutil
import os
from google.colab import drive

# 1. Mount Google Drive
drive.mount('/content/drive')

# 2. Define source (Colab) and destination (Drive)
source_folder = '/content/'
# Change 'Colab_Backup_Folder' to whatever you want the folder to be named in Drive
destination_folder = '/content/drive/MyDrive/Colab_Backup_Folder'

# 3. Copy the contents
# We ignore hidden folders like .config or .ipynb_checkpoints to keep it clean
def copy_contents(src, dst):
    if not os.path.exists(dst):
        os.makedirs(dst)

    for item in os.listdir(src):
        s = os.path.join(src, item)
        d = os.path.join(dst, item)

        # Skip Google Drive itself and system folders to avoid infinite loops
        if item in ['drive', '.config', 'sample_data']:
            continue

        if os.path.isdir(s):
            shutil.copytree(s, d, dirs_exist_ok=True)
        else:
            shutil.copy2(s, d)

copy_contents(source_folder, destination_folder)

print(f"All files have been copied directly to: {destination_folder}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
All files have been copied directly to: /content/drive/MyDrive/Colab_Backup_Folder


In [ ]:
# ============================================================
# Cell 11: Evaluate on MATH500
# ============================================================
print("Loading MATH500 test set...")

math500_problems = []
math500_answers = []

try:
    m500 = load_dataset("HuggingFaceH4/MATH-500", split="test")
    for ex in m500:
        math500_problems.append(ex["problem"])
        ans = ex.get("answer", "") or extract_boxed(ex.get("solution", ""))
        math500_answers.append(ans)
    print(f"  Loaded {len(math500_problems)} MATH500 problems from HF")
except Exception as e:
    print(f"  HF MATH500 unavailable: {e}")
    try:
        m_test = load_dataset("hendrycks/competition_math", split="test")
        for ex in list(m_test)[:500]:
            math500_problems.append(ex["problem"])
            math500_answers.append(extract_boxed(ex.get("solution", "")))
        print(f"  Loaded {len(math500_problems)} problems from MATH test")
    except Exception as e2:
        print(f"  MATH test also unavailable: {e2}")

# Load model for evaluation if needed
if 'model' not in dir() or model is None:
    print("\nLoading merged model...")
    model = AutoModelForCausalLM.from_pretrained(
        MERGED_DIR,
        torch_dtype=torch.float16,
        device_map="auto",
        trust_remote_code=True,
    )
model.eval()

if math500_problems:
    math500_acc, math500_results = evaluate_dataset(
        model, tokenizer, math500_problems, math500_answers, "MATH500"
    )
else:
    print("No MATH500 data available for evaluation.")
    math500_acc = 0.0

Loading MATH500 test set...


README.md:   0%|          | 0.00/412 [00:00<?, ?B/s]

test.jsonl: 0.00B [00:00, ?B/s]

Generating test split:   0%|          | 0/500 [00:00<?, ? examples/s]

  Loaded 500 MATH500 problems from HF

Evaluating on MATH500 (500 problems)...


MATH500:   0%|          | 0/500 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [ ]:
# ============================================================
# Cell 12: Evaluate on GSM8K Test Set
# ============================================================
print("Loading GSM8K test set...")

gsm8k_problems = []
gsm8k_answers = []

try:
    gsm_test = load_dataset("openai/gsm8k", "main", split="test")
    for ex in gsm_test:
        gsm8k_problems.append(ex["question"])
        gsm8k_answers.append(extract_gsm8k_answer(ex["answer"]))
    print(f"  Loaded {len(gsm8k_problems)} GSM8K test problems")
except Exception as e:
    print(f"  GSM8K test unavailable: {e}")

if gsm8k_problems:
    gsm8k_acc, gsm8k_results = evaluate_dataset(
        model, tokenizer, gsm8k_problems, gsm8k_answers, "GSM8K"
    )
else:
    gsm8k_acc = 0.0

In [ ]:
# ============================================================
# Cell 13: Setup One-Shot-RLVR Evaluation Pipeline
# ============================================================
print("Setting up One-Shot-RLVR evaluation pipeline...")
print("This evaluates on AIME24, AIME25, AMC23, Minerva, OlympiadBench\n")

!git clone https://github.com/ypwang61/One-Shot-RLVR.git /content/One-Shot-RLVR 2>/dev/null || echo "Repo already cloned"
!cd /content/One-Shot-RLVR/Qwen2.5-Eval/evaluation/latex2sympy && pip install -e . -q 2>/dev/null
!pip install -q pebble timeout-decorator word2number 2>/dev/null
print("RLVR evaluation pipeline ready.")

In [ ]:
# ============================================================
# Cell 14: Run RLVR Evaluation Script on DAPO Model
# ============================================================
import os

eval_dir = "/content/One-Shot-RLVR/Qwen2.5-Eval/evaluation"
dapo_eval_output = "/content/eval_results/DAPO-Qwen2.5-Math-1.5B"

eval_script = f'''PROMPT_TYPE="qwen25-math-cot"
export CUDA_VISIBLE_DEVICES="0"
MAX_TOKENS="3072"

echo "======== Evaluating DAPO-trained model ========"
MODEL_NAME_OR_PATH="{MERGED_DIR}"
OUTPUT_DIR="{dapo_eval_output}"

bash sh/eval_all_math.sh $PROMPT_TYPE $MODEL_NAME_OR_PATH $MAX_TOKENS $OUTPUT_DIR
'''

script_path = os.path.join(eval_dir, "sh/eval_dapo.sh")
os.makedirs(os.path.dirname(script_path), exist_ok=True)
with open(script_path, 'w') as f:
    f.write(eval_script)

print("DAPO evaluation script created.")
print("Running evaluation (this may take ~20-30 min per benchmark)...\n")

# Note: This requires vllm. If it fails, MATH500/GSM8K results above are valid.
%cd {eval_dir}
!WANDB_MODE=disabled bash sh/eval_dapo.sh

In [ ]:
# ============================================================
# Cell 15: Results Comparison & Visualization
# ============================================================

# RLVR baseline results (from your RLVR.ipynb)
rlvr_results = {
    "AIME25x8": 5.0,
    "AMC23x8": 50.9,
    "AIME24x8": 12.5,
    "Minerva": 27.9,
    "OlympiadBench": 33.8,
    "MATH500": 73.0,
}

# DAPO results
dapo_results = {
    "MATH500": math500_acc if 'math500_acc' in dir() else 0.0,
}

# Parse RLVR-pipeline results if available
eval_output_dir = "/content/eval_results/DAPO-Qwen2.5-Math-1.5B"
key_map = {
    "aime25x8": "AIME25x8", "amc23x8": "AMC23x8",
    "aime24x8": "AIME24x8", "minerva_math": "Minerva",
    "olympiadbench": "OlympiadBench", "math500": "MATH500"
}
for benchmark in key_map:
    result_dir = os.path.join(eval_output_dir, benchmark)
    if os.path.isdir(result_dir):
        for fname in os.listdir(result_dir):
            if fname.endswith('.jsonl'):
                try:
                    with open(os.path.join(result_dir, fname)) as fp:
                        lines = fp.readlines()
                    if lines:
                        last = json.loads(lines[-1])
                        if 'acc' in last:
                            dapo_results[key_map[benchmark]] = last['acc']
                except Exception:
                    pass

# Print comparison table
print("\n" + "=" * 70)
print("  RESULTS COMPARISON: RLVR vs DAPO")
print("=" * 70)
print(f"{'Benchmark':<16} {'RLVR':>10} {'DAPO':>10} {'Delta':>10}")
print("-" * 46)

rlvr_avg, dapo_avg, count = 0, 0, 0
for bench in ["MATH500", "AIME25x8", "AMC23x8", "AIME24x8", "Minerva", "OlympiadBench"]:
    rlvr_val = rlvr_results.get(bench)
    dapo_val = dapo_results.get(bench)
    delta = ""
    marker = ""
    if rlvr_val is not None and dapo_val is not None and dapo_val > 0:
        diff = dapo_val - rlvr_val
        delta = f"{'+' if diff >= 0 else ''}{diff:.1f}%"
        marker = " >>>" if diff > 0 else ""
        rlvr_avg += rlvr_val
        dapo_avg += dapo_val
        count += 1

    rlvr_str = f"{rlvr_val:.1f}%" if rlvr_val else "N/A"
    dapo_str = f"{dapo_val:.1f}%" if dapo_val and dapo_val > 0 else "N/A"
    print(f"{bench:<16} {rlvr_str:>10} {dapo_str:>10} {delta:>10}{marker}")

if count > 0:
    print("-" * 46)
    rlvr_avg /= count
    dapo_avg /= count
    diff = dapo_avg - rlvr_avg
    marker = " >>>" if diff > 0 else ""
    print(f"{'Average':<16} {rlvr_avg:>9.1f}% {dapo_avg:>9.1f}% {'+' if diff >= 0 else ''}{diff:.1f}%{marker}")
print("=" * 70)
if 'gsm8k_acc' in dir() and gsm8k_acc > 0:
    print(f"\nGSM8K Test Accuracy: {gsm8k_acc:.1f}%")

# Visualization
benchmarks_to_plot = []
rlvr_vals = []
dapo_vals = []
for b in ["MATH500", "AMC23x8", "OlympiadBench", "Minerva", "AIME24x8", "AIME25x8"]:
    if b in rlvr_results and b in dapo_results and dapo_results[b] > 0:
        benchmarks_to_plot.append(b)
        rlvr_vals.append(rlvr_results[b])
        dapo_vals.append(dapo_results[b])

if benchmarks_to_plot:
    fig, ax = plt.subplots(figsize=(10, 5))
    x = np.arange(len(benchmarks_to_plot))
    w = 0.35
    bars1 = ax.bar(x - w/2, rlvr_vals, w, label='RLVR Baseline', color='#5b9bd5', edgecolor='white')
    bars2 = ax.bar(x + w/2, dapo_vals, w, label='DAPO (Ours)', color='#ed7d31', edgecolor='white')
    ax.set_ylabel('Accuracy (%)', fontsize=12)
    ax.set_title('RLVR vs DAPO: Math Benchmark Comparison', fontsize=14, fontweight='bold')
    ax.set_xticks(x)
    ax.set_xticklabels(benchmarks_to_plot, rotation=15, ha='right')
    ax.legend(fontsize=11)
    ax.set_ylim(0, max(max(rlvr_vals), max(dapo_vals)) * 1.15)
    ax.grid(axis='y', alpha=0.3)
    for bar in bars1:
        ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.5,
                f'{bar.get_height():.1f}', ha='center', va='bottom', fontsize=9)
    for bar in bars2:
        ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.5,
                f'{bar.get_height():.1f}', ha='center', va='bottom', fontsize=9)
    plt.tight_layout()
    plt.savefig('/content/dapo_vs_rlvr_comparison.png', dpi=150)
    plt.show()
    print("\nChart saved to /content/dapo_vs_rlvr_comparison.png")
else:
    print("\nNo benchmark comparison data available for plotting.")

## Summary

This notebook implemented **DAPO** to improve upon One-Shot-RLVR:

### DAPO Techniques Applied:
1. **Clip-Higher** (epsilon=0.28): Relaxed upper clipping for exploration
2. **Dynamic Sampling**: GRPO naturally skips zero-signal batches
3. **Token-Level Loss**: Per-token gradient removes length bias
4. **Overlong Reward Shaping**: Soft penalty for verbose responses

### Tips to Further Improve:
- **More steps**: Increase `TRAIN_STEPS` to 500-1000
- **Larger dataset**: Use full DeepScaleR (40k) or MATH+GSM8K combined
- **Better GPU**: A100/L4 allows larger group size (G=8 or 16)
- **Multi-epoch**: Run multiple passes over training data
- **Reward engineering**: Add step verification rewards